# EDA analysis notebook

This notebook stands for EDA of 
[this dataset](https://www.kaggle.com/competitions/rossmann-store-sales/data?select=train.csv) (Rossmann Store Sales competition)

## Some notes about dataset

- sales data for 1,115 Rossmann stores
- Forecasting horizon is about 6 weeks (from competition requirements)
- Reliable sales forecasts enable store managers to create effective staff schedules that increase productivity and motivation
- Some stores in the dataset were temporarily closed for refurbishment.
- Note that all schools are closed on public holidays and weekends

## Prelimitary analysis

In [ ]:
from google.colab import drive
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/data/store-sales-forecasting/train.csv')
df = df.set_index('Date')
df.index = pd.to_datetime(df.index)

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
rows, columns = df.shape

print(f"Rows: {rows}\nColumns: {columns}")

In [ ]:
print(f"total number of stores is {df["Store"].unique().sum()}")

In [ ]:
df.isna().sum()

In [ ]:
print(f'Duplicated rows: {df.index.duplicated().sum()} ({df.index.duplicated().sum() / rows*100:.2f}%)')

as we have multiple stores at 1 day (each row stands for day and store) we have many dublicates in dataset

## Feature preparation (basic)

Extract total sales, date, day of a week and e.c from the dataset

In [ ]:
df["Total sales"] = df.groupby("Date")["Sales"].sum()

In [ ]:
df[~df.index.duplicated()].head()

In [ ]:
df["Day of month"] = df.index.day
df["Month"] = df.index.month
df["Year"] = df.index.year
df["Week of year"] = df.index.isocalendar().week.astype(int)
df["Day of year"] = df.index.dayofyear
df["Quarter"] = df.index.quarter
df["Is weekend"] = (df.index.dayofweek >= 5).astype(int)

def get_season(month):
    if month in [12, 1, 2]:
        return 0  # Winter
    elif month in [3, 4, 5]:
        return 1  # Spring
    elif month in [6, 7, 8]:
        return 2  # Summer
    else:
        return 3  # Autumn

df["Season"] = df["Month"].apply(get_season)

In [ ]:
df.columns

In [ ]:
df.head()

## Distribution analysis

Plot the distributions to meet my future work

In [ ]:
def clear_x_labels(ax):
    """Delete x labels except the last one
    Useful if plots shares same X label
    """
    for a in ax[:-1]:
        a.set_xlabel("")
    return ax

def plot_time_series_plots(nrows,ncolumns, df):
    fig, ax = plt.subplots(nrows,ncolumns, figsize=(20,10))
    fig.subplots_adjust(hspace=0.5)
    
    agg_df = df.groupby("Date")["Sales"].agg("sum")
    
    agg_df.plot(title="Sales over time", ax=ax[0])
    agg_df["2013-01-01" : "2013-12-31"].plot(title="Sales over time (2013-2014)", ax=ax[1])
    agg_df["2013-04-01" : "2013-04-30"].plot(title="Sales over time (1 month, April)", ax=ax[2])
    agg_df["2013-02-01" : "2013-02-8"].plot(title="Sales over time (1 week, 1-8 of February)", ax=ax[3])
    
    ax = clear_x_labels(ax)
    
plot_time_series_plots(4,1,df)
plt.show()

In [ ]:
# TODO there is an outlier at january 2014
# TODO also there is similar otlier
# TODO: week plot is not similar to patterns

In [ ]:
store_sales = df.groupby("Store")["Sales"].sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

store_sales.plot(kind="hist", bins=75, ax=axes[0], edgecolor="black")
axes[0].set_title("Distribution of Total Sales per Store")
axes[0].set_xlabel("Total Sales")
axes[0].set_ylabel("Number of Stores")

top_n = 20
store_sales.tail(top_n).plot(kind="barh", ax=axes[1], color="steelblue")
axes[1].set_title(f"Top {top_n} Stores by Total Sales")
axes[1].set_xlabel("Total Sales")
axes[1].set_ylabel("Store")

plt.tight_layout()
plt.show()

In [ ]:
open_df = df[df["Open"] == 1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(open_df["Sales"], bins=60, edgecolor="black", alpha=0.7)
axes[0].set_title("Sales Distribution (Open Stores)")
axes[0].set_xlabel("Sales")
axes[0].set_ylabel("Frequency")

axes[1].hist(open_df["Customers"], bins=60, edgecolor="black", alpha=0.7, color="steelblue")
axes[1].set_title("Customers Distribution (Open Stores)")
axes[1].set_xlabel("Customers")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
open_df = df[df["Open"] == 1].copy()
season_labels = {0: "Winter", 1: "Spring", 2: "Summer", 3: "Autumn"}
open_df["SeasonLabel"] = open_df["Season"].map(season_labels)

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

sns.boxplot(data=open_df, x="DayOfWeek", y="Sales", ax=axes[0])
axes[0].set_title("Sales by Day of Week")

sns.boxplot(data=open_df, x="Promo", y="Sales", ax=axes[1])
axes[1].set_title("Sales by Promo")
axes[1].set_xticklabels(["No Promo", "Promo"])

sns.boxplot(data=open_df, x="SchoolHoliday", y="Sales", ax=axes[2])
axes[2].set_title("Sales by School Holiday")
axes[2].set_xticklabels(["No Holiday", "Holiday"])

sns.boxplot(data=open_df, x="Month", y="Sales", ax=axes[3])
axes[3].set_title("Sales by Month")

sns.boxplot(data=open_df, x="SeasonLabel", y="Sales", ax=axes[4],
            order=["Winter", "Spring", "Summer", "Autumn"])
axes[4].set_title("Sales by Season")

axes[5].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=open_df, x="StateHoliday", y="Sales", ax=axes[0])
axes[0].set_title("Sales by State Holiday")

sns.boxplot(data=open_df, x="Is weekend", y="Sales", ax=axes[1])
axes[1].set_title("Sales by Weekend")
axes[1].set_xticklabels(["Weekday", "Weekend"])

axes[2].scatter(open_df["Customers"], open_df["Sales"], alpha=0.05, s=5)
axes[2].set_title("Customers vs Sales")
axes[2].set_xlabel("Customers")
axes[2].set_ylabel("Sales")

plt.tight_layout()
plt.show()